# Requirements

## Standard libraries

In [1]:
import numpy as np
import pandas as pd
import networkx as nx

from collections import defaultdict
from itertools import chain, combinations
from typing import Callable, NamedTuple, Sequence

import json
from pathlib import Path
import pickle

import numpy as np
import networkx as nx
from rpy2.robjects import r

from time import perf_counter
from sklearn.preprocessing import StandardScaler

from causallearn.utils.cit import CIT
from causallearn.utils.Dataset import load_dataset

from time import perf_counter
from tqdm import tqdm

## R libraries

In [2]:
import rpy2.robjects as robjects
import os

# 5. Verify core dependencies mentioned in the paper
try:
    robjects.r('library(pcalg)')
    robjects.r('library(igraph)')
    robjects.r('library(bnlearn)')
    print("✅ R environment for LOAD is ready!")
except Exception as e:
    print(f"❌ Failed to load library: {e}")

R callback write-console: 
Attaching package: ‘igraph’

  
R callback write-console: The following objects are masked from ‘package:stats’:

    decompose, spectrum

  
R callback write-console: The following object is masked from ‘package:base’:

    union

  
R callback write-console: 
Attaching package: ‘bnlearn’

  
R callback write-console: The following objects are masked from ‘package:igraph’:

    as.igraph, compare, degree, subgraph

  
R callback write-console: The following objects are masked from ‘package:pcalg’:

    dsep, pdag2dag, shd, skeleton

  


✅ R environment for LOAD is ready!


# Functions

## Data Generation

### Main generation script

In [3]:
import json
from pathlib import Path
import pickle

import numpy as np
import networkx as nx
from rpy2.robjects import r



def save_data(data: dict, seed: int, **kwargs):
    """
    Save experimental data to a file identified by the seed and the kwargs.

    Args:
        data (dict): The experimental data to save.
        seed (int): The seed used to generate the data.
        **kwargs: Additional arguments used to generate the data.
    """
    file_path = Path("experiments/{}/{}.pkl".format(json.dumps(kwargs), seed))
    file_path.parent.mkdir(parents=True, exist_ok=True)
    with file_path.open("wb") as file:
        pickle.dump(data, file)


def load_data(seed: int, **kwargs) -> dict:
    """´
    Load experimental data from a file identified by the seed and the kwargs.

    Args:
        seed (int): The seed used to generate the data.
        **kwargs: Additional arguments used to generate the data.

    Returns:
        dict: The loaded experimental data.
    """
    file_path = Path("experiments/{}/{}.pkl".format(json.dumps(kwargs), seed))
    with file_path.open("rb") as file:
        return pickle.load(file)


def generate_data(
    seed: int,
    observed: int,
    exp_degree: float,
    max_degree: int,
    targets: int,
    latent: int = 0,
    connected: bool = True,
    expl_anc: bool = False,
    identifiable: bool = False,
    min_adj_size: int = 0,
    samples_num: int = 0,
    discrete: bool = False,
    max_classes: int = 2,
    file: str | None = None,
    save: bool = True,
) -> dict:
    """
    Generate experimental data given the parameters.

    Args:
        seed (int): The seed for the random number generator.
        file (str | None): The file to load data from.
        observed (int): Number of observed nodes.
        latent (int): Number of latent nodes.
        exp_degree (float): Expected degree of the graph.
        max_degree (int): Maximum degree of the graph.
        targets (int): Number of target nodes.
        connected (bool): Whether the graph should be connected.
        expl_anc (bool): Whether the targets should be an explicit treatment-outcome pair.
        identifiable (bool): Whether the graph should be identifiable.
        min_adj_size (int): Minimum size of the adjacency set.
        samples_num (int): Number of samples.
        discrete (bool): Whether the data is discrete.
        max_classes (int): Maximum number of classes for discrete variables.

    Returns:
        dict: The generated experimental data.
    """
    if file is None:  # Generate causal model from scratch
        gen_func = r["generate_data"]
        kwargs = {
            "seed": seed,
            "observed": observed,
            "latent": latent,
            "exp_degree": exp_degree,
            "max_degree": max_degree,
            "targets": targets,
            "connected": connected,
            "expl_anc": expl_anc,
            "identifiable": identifiable,
            "min_adj_size": min_adj_size,
            "samples_num": samples_num,
            "discrete": discrete,
            "max_classes": max_classes,
        }
    else:  # Generate data according to model from file
        gen_func = r["generate_data_from_file"]
        kwargs = {
            "file": file,
            "seed": seed,
            "targets": targets,
            "identifiable": identifiable,
            "min_adj_size": min_adj_size,
            "samples_num": samples_num,
        }
    try:  # Try to load previously generated experimental data
        return load_data(**kwargs)
    except FileNotFoundError:  # Generate new experimental data
        r_data = gen_func(**kwargs)
        true_dag = r_data.rx2("suffStat").rx2("g")
        true_dag = nx.from_numpy_array(
            np.array(r["as"](true_dag, "matrix")),
            create_using=nx.DiGraph,
        )
        dm = np.array(r_data.rx2("suffStat").rx2("dm"))
        data = {
            "id": r_data.rx2("id")[0],
            "data": dm,
            "targets": np.array(r_data.rx2("targets")).astype(np.int32) - 1,
            "true_dag": true_dag,
            "cpt": r_data.rx2("cpt"),
            "treatment": np.array(r_data.rx2("treatment")).astype(np.int32)[0] - 1,
            "outcome": np.array(r_data.rx2("outcome")).astype(np.int32)[0] - 1,
        }
        if save:
            save_data(data, **kwargs)
        return data

In [4]:
# 1. Define the absolute path to your shared folder shortcut or MyDrive folder
# Use the 'Copy Path' feature in the Colab sidebar to get this exactly right
drive_path = "generate_data.R"

# 2. Source the file using the R 'source' function via robjects
if os.path.exists(drive_path):
    robjects.r.source(drive_path)
    print(f"Successfully sourced: {drive_path}")
else:
    print(f"Error: File not found at {drive_path}")

Successfully sourced: generate_data.R


### Background knowledge sampler

In [5]:
import random
import networkx as nx
import math
import numpy as np

def sample_local_background_knowledge(
    true_dag: nx.DiGraph,
    targets: np.ndarray,
    fraction_required: float = 0.1,
    fraction_forbidden: float = 0.1,
    seed: int = 42
):
    """
    Samples required and forbidden edges specifically connected to target nodes.
    """
    random.seed(seed)
    nodes = list(true_dag.nodes())
    target_set = set(targets)
    true_edges = list(true_dag.edges())

    # 1. Identify "Local" True Edges (pointing to or from targets)
    local_true_edges = [
        (u, v) for u, v in true_edges
        if u in target_set or v in target_set
    ]

    # 2. Identify "Local" Non-Edges
    # Every possible pair involving at least one target minus existing edges
    all_possible_local = [
        (u, v) for u in nodes for v in nodes
        if u != v and (u in target_set or v in target_set)
    ]
    local_non_edges = [e for e in all_possible_local if e not in set(true_edges)]

    # Helper to calculate count with a minimum of 1 if fraction > 0
    def get_sample_count(pool, fraction):
        if fraction <= 0 or not pool:
            return 0
        return max(1, math.ceil(len(pool) * fraction))

    # Sampling
    num_req = get_sample_count(local_true_edges, fraction_required)
    num_forb = get_sample_count(local_non_edges, fraction_forbidden)

    required = set(random.sample(local_true_edges, num_req)) if num_req > 0 else set()
    forbidden = set(random.sample(local_non_edges, num_forb)) if num_forb > 0 else set()

    return {"forbidden": forbidden, "required": required}

# Testing required edges quantity

In [7]:
def test_required_edges_quantity(
        seed,
        observed,
        exp_degree,
        max_degree,
        targets,
        identifiable,
        min_adj_size,
        samples_num,
        expl_anc,
        discrete,
        save,
        fraction_required,
        fraction_forbidden,
):
    # Generate the synthetic data
    data = generate_data(
        seed=seed,
        observed=observed,        # Slightly more nodes to ensure a richer MB
        exp_degree=exp_degree,
        max_degree=max_degree,
        targets=targets,
        identifiable=identifiable,
        min_adj_size=min_adj_size,
        samples_num=samples_num,   # More samples for more reliable Fisher-Z tests
        expl_anc=expl_anc,
        discrete=discrete,
        save=save
    )


    # Extract components
    true_dag = data["true_dag"]
    cpt = data["cpt"]
    targets = data["targets"]


    # Sample background knowledge (e.g., 20% of edges)
    bg_knowledge = sample_local_background_knowledge(true_dag, targets, fraction_required, fraction_forbidden, seed)

    return len(bg_knowledge["required"])

In [ ]:
# ER2 sample size 1000

results = []
configs = [
    (observed, fraction_required)
    for observed in [10, 15, 20, 25]
    for fraction_required in [0.2, 0.3, 0.4, 0.5]
]

total_runs = len(configs) * 100
pbar = tqdm(total=total_runs, desc="Total required-edge evaluations")
base_seed = 42

for idx, (observed, fraction_required) in enumerate(configs):
    required_counts = []

    for rep in range(100):
        seed = base_seed + idx * 100 + rep
        required_count = test_required_edges_quantity(
            seed=seed,
            observed=observed,
            exp_degree=2.0,
            max_degree=5,
            targets=2,
            identifiable=True,
            min_adj_size=1,
            samples_num=1000,
            expl_anc=False,
            discrete=False,
            save=False,
            fraction_required=fraction_required,
            fraction_forbidden=0,
        )
        required_counts.append(required_count)
        pbar.update(1)

    avg_required_edges = np.mean(required_counts)
    results.append(
        {
            "observed": observed,
            "fraction_required": fraction_required,
            "average_required_edges": avg_required_edges,
        }
    )

pbar.close()




Total required-edge evaluations:   3%|▎         | 47/1600 [00:13<08:38,  3.00it/s]

In [ ]:
import pandas as pd

df = pd.DataFrame(results, columns=["observed", "fraction_required", "average_required_edges"])
df


,observed,fraction_required,average_required_edges
0,10,0.2,1.58
1,10,0.3,2.35
2,10,0.4,2.65
3,10,0.5,3.17
4,15,0.2,1.66
5,15,0.3,2.33
6,15,0.4,2.72
7,15,0.5,3.25
8,20,0.2,1.58
9,20,0.3,2.41


In [ ]:
# ER3 sample size 1000

results = []
configs = [
    (observed, fraction_required)
    for observed in [10, 15, 20, 25]
    for fraction_required in [0.2, 0.3, 0.4, 0.5]
]

total_runs = len(configs) * 100
pbar = tqdm(total=total_runs, desc="Total required-edge evaluations")
base_seed = 42

for idx, (observed, fraction_required) in enumerate(configs):
    required_counts = []

    for rep in range(100):
        seed = base_seed + idx * 100 + rep
        required_count = test_required_edges_quantity(
            seed=seed,
            observed=observed,
            exp_degree=3.0,
            max_degree=7,
            targets=2,
            identifiable=True,
            min_adj_size=1,
            samples_num=1000,
            expl_anc=False,
            discrete=False,
            save=False,
            fraction_required=fraction_required,
            fraction_forbidden=0,
        )
        required_counts.append(required_count)
        pbar.update(1)

    avg_required_edges = np.mean(required_counts)
    results.append(
        {
            "observed": observed,
            "fraction_required": fraction_required,
            "average_required_edges": avg_required_edges,
        }
    )

pbar.close()


df2 = pd.DataFrame(results, columns=["observed", "fraction_required", "average_required_edges"])
df2


Total required-edge evaluations: 100%|██████████| 1600/1600 [08:31<00:00,  3.13it/s]


,observed,fraction_required,average_required_edges
0,10,0.2,1.84
1,10,0.3,2.63
2,10,0.4,3.22
3,10,0.5,3.81
4,15,0.2,1.90
5,15,0.3,2.73
6,15,0.4,3.36
7,15,0.5,3.91
8,20,0.2,1.96
9,20,0.3,2.83


In [ ]:
# ER3 sample size 1000

results = []
configs = [
    (observed, fraction_required)
    for observed in [10, 15, 20, 25]
    for fraction_required in [0.2, 0.3, 0.4, 0.5]
]

total_runs = len(configs) * 100
pbar = tqdm(total=total_runs, desc="Total required-edge evaluations")
base_seed = 42

for idx, (observed, fraction_required) in enumerate(configs):
    required_counts = []

    for rep in range(100):
        seed = base_seed + idx * 100 + rep
        required_count = test_required_edges_quantity(
            seed=seed,
            observed=observed,
            exp_degree=4.0,
            max_degree=8,
            targets=2,
            identifiable=True,
            min_adj_size=1,
            samples_num=1000,
            expl_anc=False,
            discrete=False,
            save=False,
            fraction_required=fraction_required,
            fraction_forbidden=0,
        )
        required_counts.append(required_count)
        pbar.update(1)

    avg_required_edges = np.mean(required_counts)
    results.append(
        {
            "observed": observed,
            "fraction_required": fraction_required,
            "average_required_edges": avg_required_edges,
        }
    )

pbar.close()


df3 = pd.DataFrame(results, columns=["observed", "fraction_required", "average_required_edges"])
df3


Total required-edge evaluations: 100%|██████████| 1600/1600 [09:55<00:00,  2.68it/s]


,observed,fraction_required,average_required_edges
0,10,0.2,2.12
1,10,0.3,3.02
2,10,0.4,3.89
3,10,0.5,4.49
4,15,0.2,2.13
5,15,0.3,3.06
6,15,0.4,3.89
7,15,0.5,4.74
8,20,0.2,2.26
9,20,0.3,3.15


# False edge generation

In [6]:
import random
import networkx as nx
import math
import numpy as np

def sample_local_background_knowledge_noised(
    true_dag: nx.DiGraph,
    targets: np.ndarray,
    fraction_required: float = 0.1,
    fraction_forbidden: float = 0.1,
    false_required_ratio: float = 0.5,
    seed: int = 42
):
    """
    Samples required and forbidden edges specifically connected to target nodes.
    Injects false required edges (starting from targets) to test robustness.
    """
    random.seed(seed)
    nodes = list(true_dag.nodes())
    target_set = set(targets)
    true_edges = list(true_dag.edges())
    true_edges_set = set(true_edges) # Set for O(1) lookups

    # 1. Identify "Local" True Edges (pointing to or from targets)
    local_true_edges = [
        (u, v) for u, v in true_edges
        if u in target_set or v in target_set
    ]

    # 2. Identify "Local" Non-Edges
    # Every possible pair involving at least one target minus existing edges
    all_possible_local = [
        (u, v) for u in nodes for v in nodes
        if u != v and (u in target_set or v in target_set)
    ]
    local_non_edges = [e for e in all_possible_local if e not in true_edges_set]

    # Helper to calculate count with a minimum of 1 if fraction > 0
    def get_sample_count(pool, fraction):
        if fraction <= 0 or not pool:
            return 0
        return max(1, math.ceil(len(pool) * fraction))

    # Sampling standard background knowledge
    num_req = get_sample_count(local_true_edges, fraction_required)
    num_forb = get_sample_count(local_non_edges, fraction_forbidden)

    required = set(random.sample(local_true_edges, num_req)) if num_req > 0 else set()
    forbidden = set(random.sample(local_non_edges, num_forb)) if num_forb > 0 else set()


    # 3. Add "False" Required Edges
    # Candidates: Edges starting from a target node that are NOT in the true DAG
    false_req_candidates = [
        (u, v) for u in target_set for v in nodes
        if u != v and (u, v) not in true_edges_set
    ]

    # Calculate quantity: 50% of num_req, minimum 1
    if false_required_ratio > 0 and false_req_candidates:
        ideal_false_req_count = max(1, math.ceil(num_req * false_required_ratio))
        # Safely cap the count so we don't try to sample more candidates than exist
        num_false_req = min(len(false_req_candidates), ideal_false_req_count)
        
        false_required = set(random.sample(false_req_candidates, num_false_req))
        required = required.union(false_required)
    return {"forbidden": forbidden, "required": required}

In [7]:
def test_required_edges_quantity_noised(
        seed,
        observed,
        exp_degree,
        max_degree,
        targets,
        identifiable,
        min_adj_size,
        samples_num,
        expl_anc,
        discrete,
        save,
        fraction_required,
        fraction_forbidden,
        false_required_ratio
):
    # Generate the synthetic data
    data = generate_data(
        seed=seed,
        observed=observed,        # Slightly more nodes to ensure a richer MB
        exp_degree=exp_degree,
        max_degree=max_degree,
        targets=targets,
        identifiable=identifiable,
        min_adj_size=min_adj_size,
        samples_num=samples_num,   # More samples for more reliable Fisher-Z tests
        expl_anc=expl_anc,
        discrete=discrete,
        save=save
    )


    # Extract components
    true_dag = data["true_dag"]
    cpt = data["cpt"]
    targets = data["targets"]


    # Sample background knowledge (e.g., 20% of edges)
    bg_knowledge = sample_local_background_knowledge_noised(true_dag, targets, fraction_required, fraction_forbidden, false_required_ratio, seed)

    return len(bg_knowledge["required"])

In [8]:
required_count = test_required_edges_quantity_noised(
    seed=42,
    observed=10,
    exp_degree=2.0,
    max_degree=5,
    targets=2,
    identifiable=True,
    min_adj_size=1,
    samples_num=1000,
    expl_anc=False,
    discrete=False,
    save=False,
    fraction_required=0.2,
    fraction_forbidden=0,
    false_required_ratio=0.5
)
required_count

2

In [9]:
# ER2 sample size 1000

results = []
configs = [
    (observed, fraction_required)
    for observed in [10, 15, 20, 25]
    for fraction_required in [0.2, 0.3, 0.4, 0.5]
]

total_runs = len(configs) * 100
pbar = tqdm(total=total_runs, desc="Total required-edge evaluations")
base_seed = 42

for idx, (observed, fraction_required) in enumerate(configs):
    required_counts = []

    for rep in range(100):
        seed = base_seed + idx * 100 + rep
        required_count = test_required_edges_quantity_noised(
            seed=seed,
            observed=observed,
            exp_degree=2.0,
            max_degree=5,
            targets=2,
            identifiable=True,
            min_adj_size=1,
            samples_num=1000,
            expl_anc=False,
            discrete=False,
            save=False,
            fraction_required=fraction_required,
            fraction_forbidden=0,
            false_required_ratio=0.5
        )
        required_counts.append(required_count)
        pbar.update(1)

    avg_required_edges = np.mean(required_counts)
    results.append(
        {
            "observed": observed,
            "fraction_required": fraction_required,
            "average_required_edges": avg_required_edges,
        }
    )

pbar.close()


df4 = pd.DataFrame(results, columns=["observed", "fraction_required", "average_required_edges"])
df4



Total required-edge evaluations: 100%|██████████| 1600/1600 [10:44<00:00,  2.48it/s]


,observed,fraction_required,average_required_edges
0,10,0.2,2.58
1,10,0.3,3.70
2,10,0.4,4.26
3,10,0.5,5.05
4,15,0.2,2.66
5,15,0.3,3.67
6,15,0.4,4.31
7,15,0.5,5.19
8,20,0.2,2.58
9,20,0.3,3.83
